# Polygon/Massive realtime prototype 
 
Este notebook construye un prototipo de ingesta en tiempo real para `stocks` usando WebSocket. La secuencia está separada en scripts dentro de `notebooks/cell_code` para que luego podamos reutilizar la misma lógica en backend web sin depender del notebook.

## Celda 0. Configuracion operativa compartida

Esta celda centraliza los parametros de ejecucion que van a reutilizar las demas celdas: modo del feed, canales, universo de simbolos, duracion y etiqueta de la sesion. El objetivo es evitar parametros hardcodeados repetidos y reducir errores al cambiar el alcance del experimento.

Motivo: en un prototipo iterativo es normal relanzar la misma secuencia cambiando solo algunos tickers o la duracion. Tener una unica fuente de verdad simplifica la ejecucion y hace mas legible el notebook.

Resultado esperado: variables Python disponibles en memoria para el resto del notebook, listas para ser pasadas a los scripts de `cell_code`.

In [1]:
from pathlib import Path

PROJECT_ROOT = Path(r"C:\TSIS_Data\v1\WebSocket_SmallCaps")
CELL_CODE_DIR = PROJECT_ROOT / "notebooks" / "cell_code"

feed_mode = "realtime"
channels = "AM,T,Q,A"
symbols = "LIDR,OCG,KNDI,LNAI"
duration_sec = 180
label = "prototype_run_phase3_A"


print({
    "feed_mode": feed_mode,
    "channels": channels,
    "symbols": symbols,
    "duration_sec": duration_sec,
    "label": label,
})


{'feed_mode': 'realtime', 'channels': 'AM,T,Q,A', 'symbols': 'LIDR,OCG,KNDI,LNAI', 'duration_sec': 180, 'label': 'prototype_run_phase3_A'}


## Celda 1. Validación técnica del runtime 
 
Este script inspecciona el runtime local antes de abrir una conexión al feed. Comprueba versión de Python, módulos críticos (`websockets`, `pandas`, `pyarrow`, `duckdb`), presencia de la API key y creación del layout base de directorios. 
 
Motivo: no tiene sentido depurar el WebSocket si el entorno ni siquiera está listo para autenticar, persistir y resumir la captura. 
 
Resultado esperado: un bloque JSON con el estado del entorno y, si falta la clave, una advertencia explícita para definir `MASSIVE_API_KEY` o `POLYGON_API_KEY`.

In [2]:
import subprocess, sys

script = CELL_CODE_DIR / "00_validate_runtime.py"

subprocess.run([sys.executable, str(script)], check=True)


CompletedProcess(args=['c:\\TSIS_Data\\v1\\WebSocket_SmallCaps\\.venv_ws\\Scripts\\python.exe', 'C:\\TSIS_Data\\v1\\WebSocket_SmallCaps\\notebooks\\cell_code\\00_validate_runtime.py'], returncode=0)

## Celda 2. Planificación de la captura 
 
Este script materializa el plan operativo de la sesión: feed (`realtime` o `delayed`), URL WebSocket efectiva, canales, símbolos, duración y rutas de salida. Además genera el string exacto de suscripción que se enviará al servidor. 
 
Motivo: fijar un plan reproducible antes de capturar evita lanzar sesiones ambiguas y deja trazabilidad de qué universo se pidió exactamente al feed. 
 
Resultado esperado: un JSON con la configuración del run y un archivo `capture_plan.json` dentro del directorio raw de la sesión.

In [3]:
import subprocess, sys

script = CELL_CODE_DIR / "01_prepare_capture_plan.py"

subprocess.run([
    sys.executable,
    str(script),
    "--feed-mode", feed_mode,
    "--channels", channels,
    "--symbols", symbols,
    "--duration-sec", str(duration_sec),
    "--label", label
], check=True)


CompletedProcess(args=['c:\\TSIS_Data\\v1\\WebSocket_SmallCaps\\.venv_ws\\Scripts\\python.exe', 'C:\\TSIS_Data\\v1\\WebSocket_SmallCaps\\notebooks\\cell_code\\01_prepare_capture_plan.py', '--feed-mode', 'realtime', '--channels', 'AM,T,Q,A', '--symbols', 'LIDR,OCG,KNDI,LNAI', '--duration-sec', '180', '--label', 'prototype_run_phase3_A'], returncode=0)

## Celda 3. Captura controlada del stream 
 
Este script abre la conexión WebSocket contra Massive/Polygon, autentica la sesión, envía la suscripción y mantiene la ingesta durante una ventana de tiempo limitada. Cada evento recibido se persiste en formato `JSONL` append-only con timestamp de captura, sin hacer transformaciones destructivas. 
 
Motivo: la primera versión del prototipo debe priorizar integridad y trazabilidad del feed recibido. Guardar el evento raw permite reparsear, depurar huecos y validar semántica de campos sin depender de decisiones tempranas de modelado. 
 
Resultado esperado: un `events.jsonl` con los eventos capturados y un `capture_meta.json` con contadores por tipo de evento, total de mensajes y parámetros de la sesión.

In [4]:
import subprocess, sys

script = CELL_CODE_DIR / "02_capture_stream.py"

subprocess.run([
    sys.executable,
    str(script),
    "--feed-mode", feed_mode,
    "--channels", channels,
    "--symbols", symbols,
    "--duration-sec", str(duration_sec),
    "--label", label
], check=True)


CompletedProcess(args=['c:\\TSIS_Data\\v1\\WebSocket_SmallCaps\\.venv_ws\\Scripts\\python.exe', 'C:\\TSIS_Data\\v1\\WebSocket_SmallCaps\\notebooks\\cell_code\\02_capture_stream.py', '--feed-mode', 'realtime', '--channels', 'AM,T,Q,A', '--symbols', 'LIDR,OCG,KNDI,LNAI', '--duration-sec', '180', '--label', 'prototype_run_phase3_A'], returncode=0)

## Celda 4. Resumen analítico de la captura 
 
Este script lee la captura raw, extrae un subconjunto operativo de campos (`ev`, `sym`, timestamp de mercado, precio y tamaño), genera una tabla tabular y la persiste en `Parquet`. También produce un resumen por tipo de evento y por símbolo. 
 
Motivo: esta fase valida que la captura es utilizable aguas abajo y deja un artefacto columnar que ya sirve para exploración, QA y futura integración con backend web. 
 
Resultado esperado: un `summary.json`, un `events.parquet` y un `events_preview.csv` con una vista corta de la captura.

In [5]:
import subprocess, sys

script = CELL_CODE_DIR / "03_summarize_capture.py"

subprocess.run([
    sys.executable,
    str(script),
    "--label", label
], check=True)


CompletedProcess(args=['c:\\TSIS_Data\\v1\\WebSocket_SmallCaps\\.venv_ws\\Scripts\\python.exe', 'C:\\TSIS_Data\\v1\\WebSocket_SmallCaps\\notebooks\\cell_code\\03_summarize_capture.py', '--label', 'prototype_run_phase3_A'], returncode=0)

## Celda 5. Inspeccion visual de la captura

Esta celda esta pensada para validacion humana directa. Lee los artefactos generados por la captura (`capture_meta.json`, `summary.json`, `events.parquet` y `events.jsonl`) y los imprime de forma legible en el notebook para comprobar que se ha descargado realmente.

Motivo: no basta con que el proceso termine con codigo 0. Aqui verificamos visualmente el volumen capturado, los tipos de evento, los simbolos presentes, las primeras filas del parquet y una muestra del raw original.

Resultado esperado: ver en pantalla si hubo solo mensajes `status` o si ya entraron eventos reales de mercado (`AM`, `A`, `T`, `Q`, etc.), junto con ejemplos concretos de los datos persistidos.

In [6]:
import subprocess, sys

script = CELL_CODE_DIR / "04_inspect_capture.py"

subprocess.run([
    sys.executable,
    str(script),
    "--label", label,
    "--head", "15"
], check=True)


CompletedProcess(args=['c:\\TSIS_Data\\v1\\WebSocket_SmallCaps\\.venv_ws\\Scripts\\python.exe', 'C:\\TSIS_Data\\v1\\WebSocket_SmallCaps\\notebooks\\cell_code\\04_inspect_capture.py', '--label', 'prototype_run_phase3_A', '--head', '15'], returncode=0)

In [7]:
result = subprocess.run(
    [
        sys.executable,
        str(script),
        "--label", label,
        "--head", "15"
    ],
    capture_output=True,
    text=True,
    check=True
)

print(result.stdout)

if result.stderr.strip():
    print("STDERR:")
    print(result.stderr)

Rutas de inspeccion
raw jsonl   : C:\TSIS_Data\v1\WebSocket_SmallCaps\data\raw_ws\prototype_run_phase3_A\events.jsonl
capture meta: C:\TSIS_Data\v1\WebSocket_SmallCaps\data\raw_ws\prototype_run_phase3_A\capture_meta.json
summary     : C:\TSIS_Data\v1\WebSocket_SmallCaps\data\curated_ws\prototype_run_phase3_A\summary.json
parquet     : C:\TSIS_Data\v1\WebSocket_SmallCaps\data\curated_ws\prototype_run_phase3_A\events.parquet

=== CAPTURE META ===
{
  "started_at_utc": "2026-03-17T11:52:45.287087+00:00",
  "finished_at_utc": "2026-03-17T11:55:46.110377+00:00",
  "feed_mode": "realtime",
  "ws_url": "wss://socket.massive.com/stocks",
  "api_key_env": "POLYGON_API_KEY",
  "channels": [
    "AM",
    "T",
    "Q",
    "A"
  ],
  "symbols": [
    "LIDR",
    "OCG",
    "KNDI",
    "LNAI"
  ],
  "subscription": "AM.LIDR,AM.OCG,AM.KNDI,AM.LNAI,T.LIDR,T.OCG,T.KNDI,T.LNAI,Q.LIDR,Q.OCG,Q.KNDI,Q.LNAI,A.LIDR,A.OCG,A.KNDI,A.LNAI",
  "duration_sec": 180,
  "total_messages": 2006,
  "total_events": 320

Qué parte estamos descargando ahora

De los 7 feeds posibles de stocks:

- T = trades
- AM = minute aggregates

No estamos descargando todavía:

- Q = quotes
- A = second aggregates
- LULD
- NOI
- FMV

Eso se ve directamente en tu capture_meta.json:

- channels: ["AM", "T"]
- subscription: "AM.LIDR,AM.OCG,T.LIDR,T.OCG"

Y además el resultado real confirma qué entró:

- T: 148
- AM: 2
- status: 5

O sea:

- sí entraron trades reales
- sí entraron 2 minute aggregates
- no entró nada de Q, A, LULD, NOI o FMV porque no los estamos pidiendo


**Qué nos falta para cerrar Fase 1 de forma limpia**

Fase 1 quedaría redonda si añadimos estas dos cosas:

- visualización explícita de tiempos en UTC y Madrid
- una celda que muestre claramente el último trade / último aggregate por ticker, no solo tablas crudas

**Qué sería pasar a Fase 2**

Entraremos en Fase 2 cuando hagamos estas mejoras estructurales:

- separar tablas por feed
    - trades
    - minute_aggs
- persistencia más seria por particiones
- reconexión automática
- métricas de latencia
- logs más operativos
- quizá DuckDB para consulta rápida